In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill
import os
import unicodedata
import re
import math
from collections import Counter
import difflib
import pandas as pd


def normalizar_localidad(texto):
    if pd.isna(texto): return "SIN REGISTRO"
    texto = str(texto).strip().upper()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    
    if "USAQUEN" in texto: return "01 - USAQUÉN"
    if "CHAPINERO" in texto: return "02 - CHAPINERO"
    if "SANTA FE" in texto: return "03 - SANTA FE"
    if "SAN CRISTOBAL" in texto: return "04 - SAN CRISTÓBAL"
    if "USME" in texto: return "05 - USME"
    if "TUNJUELITO" in texto: return "06 - TUNJUELITO"
    if "BOSA" in texto: return "07 - BOSA"
    if "KENNEDY" in texto: return "08 - KENNEDY"
    if "FONTIBON" in texto: return "09 - FONTIBÓN"
    if "ENGATIVA" in texto: return "10 - ENGATIVÁ"
    if "SUBA" in texto: return "11 - SUBA"
    if "BARRIOS UNIDOS" in texto: return "12 - BARRIOS UNIDOS"
    if "TEUSAQUILLO" in texto: return "13 - TEUSAQUILLO"
    if "MARTIRES" in texto: return "14 - LOS MÁRTIRES"
    if "ANTONIO NARINO" in texto: return "15 - ANTONIO NARIÑO"
    if "PUENTE ARANDA" in texto: return "16 - PUENTE ARANDA"
    if "CANDELARIA" in texto: return "17 - LA CANDELARIA"
    if "RAFAEL URIBE" in texto: return "18 - RAFAEL URIBE URIBE"
    if "CIUDAD BOLIVAR" in texto: return "19 - CIUDAD BOLÍVAR"
    if "SUMAPAZ" in texto: return "20 - SUMAPAZ"
    
    return texto.replace("ALCALDIA LOCAL DE ", "").strip()

# ==========================================
# CLASE NLP: MOTOR DE RECUPERACIÓN RANQUEADA
# (Se mantiene exactamente igual, sin cambios)
# ==========================================
class MotorRecuperacion:
    def __init__(self, stop_words=None, n_max=3): 
        self.stop_words = stop_words if stop_words else set()
        self.n_max = n_max 
        self.idf = {}
        self.tf_idf_enciclopedia = [] 
        self.magnitudes_enciclopedia = []
        self.etiquetas_categorias = []
        self.vocabulario_conocido = set() 

    def normalizar(self, texto):
        if not texto: return ""
        texto = ''.join(c for c in unicodedata.normalize('NFD', str(texto)) if unicodedata.category(c) != 'Mn')
        texto = texto.lower()
        texto = re.sub(r'[^a-z0-9\s]', '', texto)
        return texto

    def corregir_ortografia(self, texto):
        palabras = self.normalizar(texto).split()
        palabras_corregidas = []
        for p in palabras:
            if p in self.stop_words:
                continue
            if p in self.vocabulario_conocido:
                palabras_corregidas.append(p)
            else:
                matches = difflib.get_close_matches(p, list(self.vocabulario_conocido), n=1, cutoff=0.75)
                if matches:
                    palabras_corregidas.append(matches[0]) 
                else:
                    palabras_corregidas.append(p) 
        return " ".join(palabras_corregidas)

    def obtener_ngramas_combinados(self, texto):
        palabras = texto.split() 
        if not palabras: return []
        ngramas = []
        rango_maximo = min(self.n_max, len(palabras))
        for n in range(1, rango_maximo + 1):
            for i in range(len(palabras) - n + 1):
                ngramas.append(" ".join(palabras[i:i+n]))
        return ngramas

    def cargar_enciclopedia(self, nombres_categorias, documentos_entrenamiento):
        self.etiquetas_categorias = nombres_categorias
        conteo_docs_con_termino = Counter()
        documentos_tokenizados = []
        
        for doc in documentos_entrenamiento:
            palabras = self.normalizar(doc).split()
            for p in palabras:
                if p not in self.stop_words:
                    self.vocabulario_conocido.add(p)
        
        for doc in documentos_entrenamiento:
            texto_limpio = " ".join([p for p in self.normalizar(doc).split() if p not in self.stop_words])
            tokens = self.obtener_ngramas_combinados(texto_limpio)
            documentos_tokenizados.append(tokens)
            for token in set(tokens):
                conteo_docs_con_termino[token] += 1
        
        N = len(documentos_entrenamiento)
        for token, freq in conteo_docs_con_termino.items():
            self.idf[token] = math.log10((1.0 + N) / freq)

        for tokens in documentos_tokenizados:
            tf_vector = Counter(tokens)
            vector_tfidf = {}
            suma_cuadrados = 0
            for token, count in tf_vector.items():
                tf_w = 1.0 + math.log10(count)
                val_tfidf = tf_w * self.idf[token]
                vector_tfidf[token] = val_tfidf
                suma_cuadrados += val_tfidf ** 2
            self.tf_idf_enciclopedia.append(vector_tfidf)
            self.magnitudes_enciclopedia.append(math.sqrt(suma_cuadrados))

    def clasificar_query(self, texto_query, umbral_similitud=0.025):
        texto_corregido = self.corregir_ortografia(texto_query)
        tokens_query = self.obtener_ngramas_combinados(texto_corregido)
        
        if not tokens_query: 
            return "-", 0.0, "Sin palabras válidas"

        tf_query = Counter(tokens_query)
        vector_query = {}
        suma_cuadrados_query = 0
        for token, count in tf_query.items():
            if token in self.idf:
                tf_w = 1.0 + math.log10(count)
                val_tfidf = tf_w * self.idf[token]
                vector_query[token] = val_tfidf
                suma_cuadrados_query += val_tfidf ** 2

        mag_query = math.sqrt(suma_cuadrados_query)
        if mag_query == 0: 
            return "-", 0.0, "Coseno = 0"

        resultados = []
        for idx, vec_cat in enumerate(self.tf_idf_enciclopedia):
            producto_punto = sum(vector_query.get(t, 0) * vec_cat.get(t, 0) for t in vec_cat)
            mag_cat = self.magnitudes_enciclopedia[idx]
            similitud = producto_punto / (mag_query * mag_cat) if mag_cat > 0 else 0
            resultados.append((self.etiquetas_categorias[idx], similitud))

        resultados.sort(key=lambda x: x[1], reverse=True)
        mejor_categoria = resultados[0][0]
        mejor_score = resultados[0][1]

        debug_str = " | ".join([f"{cat}: {score:.3f}" for cat, score in resultados if score > 0])
        if not debug_str:
            debug_str = "0.0 en todas"

        if mejor_score >= umbral_similitud:
            return mejor_categoria, mejor_score, debug_str
        return "-", mejor_score, debug_str

# ==========================================
# INTERFAZ Y PROCESAMIENTO EXCEL (RESTAURADO)
# ==========================================
def normalizar_texto(texto):
    if texto is None:
        return ""
    texto = str(texto).strip().upper()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[\s_\-]+', '', texto)
    return texto

def procesar_excel():
    root = tk.Tk()
    root.withdraw()
    
    print("Paso 1: Seleccionando archivo TXT de categorías...")
    ruta_txt = filedialog.askopenfilename(
        title="Selecciona archivo de Categorías (.txt)",
        filetypes=[("Archivos de Texto", "*.txt")]
    )
    if not ruta_txt: return

    print("Paso 1.1: Seleccionando archivo TXT de Clústeres...")
    ruta_cluster = filedialog.askopenfilename(
        title="Selecciona archivo de Clústeres (.txt)", 
        filetypes=[("Archivos de Texto", "*.txt")]
    )
    dict_clusters = {}
    if ruta_cluster:
        with open(ruta_cluster, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    # Intenta separar por tabulador, pipe, punto y coma o coma
                    for sep in ['\t', '|', ';', ',']:
                        if sep in line:
                            partes = line.split(sep)
                            dict_clusters[partes[0].strip()] = partes[1].strip()
                            break
    
    print("Paso 1.2: Seleccionando Excel de mapeo de Clústeres 2...")
    ruta_cluster2 = filedialog.askopenfilename(title="Archivo Clústeres 2 (.xlsx)", filetypes=[("Archivos de Texto", "*.txt")])
    dict_clusters_2 = {}
    if ruta_cluster2:
            with open(ruta_cluster2, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        # Intenta separar por tabulador, pipe, punto y coma o coma
                        for sep in ['\t', '|', ';', ',']:
                            if sep in line:
                                partes = line.split(sep)
                                dict_clusters_2[partes[0].strip()] = partes[1].strip()
                                break

    print("Paso 1.3: Seleccionando archivo TXT con lista de conteo (Tercera Hoja)...")
    ruta_txt_lista = filedialog.askopenfilename(
        title="Selecciona archivo Lista Específica (.txt)", 
        filetypes=[("Archivos de Texto", "*.txt")]
    )
    lista_conteo_personalizada = []
    if ruta_txt_lista:
        with open(ruta_txt_lista, 'r', encoding='utf-8') as f:
            lista_conteo_personalizada = [line.strip().upper() for line in f if line.strip()]

    nombres_categorias = []
    documentos_entrenamiento = []
    
    try:
        with open(ruta_txt, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    if '|' in line:
                        partes = line.split('|')
                        etiqueta = partes[0].strip()
                        sinonimos = partes[1].strip()
                        nombres_categorias.append(etiqueta)
                        documentos_entrenamiento.append(etiqueta + " " + sinonimos)
                    else:
                        nombres_categorias.append(line.strip())
                        documentos_entrenamiento.append(line.strip())
    except Exception as e:
        print(f"Error al leer TXT: {e}")
        return

    # Entrenar el motor
    stop_words_espanol = {'de', 'la', 'el', 'en', 'y', 'que', 'los', 'las', 'un', 'una', 'al', 'por', 'con', 'para'}
    try:
        with open("Stopwords-es.txt", "r", encoding="utf-8") as archivo:
            stop_words_espanol = set([linea.strip() for linea in archivo if linea.strip()])
    except:
        print("Aviso: No se encontró 'Stopwords-es.txt', usando stop words por defecto.")
        
    motor = MotorRecuperacion(stop_words=stop_words_espanol, n_max=3)
    motor.cargar_enciclopedia(nombres_categorias, documentos_entrenamiento)

    print("Paso 2: Seleccionando archivo Excel a procesar...")
    ruta_archivo = filedialog.askopenfilename(
        title="Selecciona el archivo Excel",
        filetypes=[("Archivos Excel", "*.xlsx")]
    )
    if not ruta_archivo: return

    try:
        print("Procesando Excel...")
        wb = openpyxl.load_workbook(ruta_archivo)
        ws = wb.active
        max_row = ws.max_row
        
        # Color amarillo para las nuevas columnas
        color_fondo = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
        
        # 1. DETECTAR EN QUÉ FILA ESTÁN LOS ENCABEZADOS
        fila_cabecera = 1
        encontrado = False
        for r in range(1, 16):
            for c in range(1, ws.max_column + 1):
                val = normalizar_texto(ws.cell(row=r, column=c).value)
                if val in ["DESCRIPCION", "FECHACIERRE", "TIEMPODESOLUCIONREAL", "ULTIMOCAMBIOASOCIADO"]:
                    fila_cabecera = r
                    encontrado = True
                    break
            if encontrado:
                break
                
        def buscar_columna(nombre_objetivo):
            nombre_limpio = normalizar_texto(nombre_objetivo)
            for col in range(1, ws.max_column + 1):
                valor_celda = ws.cell(row=fila_cabecera, column=col).value
                if normalizar_texto(valor_celda) == nombre_limpio:
                    return col
            return None

        columnas_no_encontradas = []

        # ==========================================
        # FASE 1: CREAR ESTRUCTURA E INSERTAR COLUMNAS
        # ==========================================
        
        # 1. ULTIMO CAMBIO ASOCIADO
        col = buscar_columna("ULTIMOCAMBIOASOCIADO")
        if col:
            ws.insert_cols(col + 1)
            ws.cell(row=fila_cabecera, column=col + 1, value="Solución").fill = color_fondo
        else:
            columnas_no_encontradas.append("ULTIMO CAMBIO ASOCIADO")

        # 2. TIEMPO DE SOLUCION REAL
        col = buscar_columna("TIEMPODESOLUCIONREAL")
        if col:
            ws.insert_cols(col + 1, amount=3)
            ws.cell(row=fila_cabecera, column=col + 1, value="Tiempos total de atención").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 2, value="formula tiempos por hora").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 3, value="Tiempos de atención por hora").fill = color_fondo
        else:
            columnas_no_encontradas.append("TIEMPO DE SOLUCION REAL")

        # 3. FECHA CIERRE
        col = buscar_columna("FECHACIERRE")
        if col:
            ws.insert_cols(col + 1)
            ws.cell(row=fila_cabecera, column=col + 1, value="Días transcurridos de solución").fill = color_fondo
        else:
            columnas_no_encontradas.append("FECHA CIERRE")

        # 4. DESCRIPCION (Adaptado para 5 columnas: NLP, Clúster 1 y Clúster 2)
        col = buscar_columna("DESCRIPCION")
        if col:
            ws.insert_cols(col + 1, amount=5)
            ws.cell(row=fila_cabecera, column=col + 1, value="Detalle Descripción").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 2, value="Score Máximo").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 3, value="Debug Scores").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 4, value="Nombre del Cluster").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 5, value="Nombre del cluster 2").fill = color_fondo
        else:
            columnas_no_encontradas.append("DESCRIPCION")

        # ==========================================
        # FASE 2: BUSCAR UBICACIONES FINALES Y APLICAR FÓRMULAS
        # ==========================================
        
        # Fórmulas de Tiempos
        col_atencion_real = buscar_columna("TIEMPODEATENCIONREAL")
        col_solucion_real = buscar_columna("TIEMPODESOLUCIONREAL")
        col_t_total       = buscar_columna("Tiempos total de atención")
        col_form_hora     = buscar_columna("formula tiempos por hora")
        col_t_hora        = buscar_columna("Tiempos de atención por hora")

        if all([col_atencion_real, col_solucion_real, col_t_total, col_form_hora, col_t_hora]):
            letra_atencion = get_column_letter(col_atencion_real)
            letra_solucion = get_column_letter(col_solucion_real)
            letra_t_total  = get_column_letter(col_t_total)

            for row in range(fila_cabecera + 1, max_row + 1):
                ws.cell(row=row, column=col_t_total, value=f"={letra_atencion}{row}+{letra_solucion}{row}").fill = color_fondo
                ws.cell(row=row, column=col_form_hora, value=f"={letra_t_total}{row}/60").fill = color_fondo
                ws.cell(row=row, column=col_t_hora, value=f"={letra_t_total}{row}/60").fill = color_fondo
        else:
            if not col_atencion_real:
                columnas_no_encontradas.append("TIEMPO DE ATENCION REAL (Necesaria para fórmulas)")

        # --- Fórmulas de Fecha Cierre ---
        col_fecha_cierre = buscar_columna("FECHACIERRE")
        col_fecha_registro = buscar_columna("FECHAREGISTRO")
        col_dias_trans = buscar_columna("Días transcurridos de solución")
        
        if col_fecha_cierre and col_fecha_registro and col_dias_trans:
            letra_fecha_cierre = get_column_letter(col_fecha_cierre)
            letra_fecha_registro = get_column_letter(col_fecha_registro)
            
            for row in range(fila_cabecera + 1, max_row + 1):
                ws.cell(row=row, column=col_dias_trans, value=f"={letra_fecha_cierre}{row}-{letra_fecha_registro}{row}").fill = color_fondo
        else:
            if not col_fecha_registro:
                columnas_no_encontradas.append("FECHA DE REGISTRO (Necesaria para fórmula de días)")

        # --- Procesamiento NLP para Descripciones ---
        col_desc_orig = buscar_columna("DESCRIPCION")
        col_detalle = buscar_columna("Detalle Descripción")
        col_cluster = buscar_columna("Nombre del Cluster")
        col_cluster2 = buscar_columna("Nombre del cluster 2")
        col_score = buscar_columna("Score Máximo")
        col_debug = buscar_columna("Debug Scores")
        
        if col_desc_orig and col_detalle and col_cluster and col_cluster2:
            for row in range(fila_cabecera + 1, max_row + 1):
                texto_query = str(ws.cell(row=row, column=col_desc_orig).value or "")
                
                cat, score, debug_str = motor.clasificar_query(texto_query)
                nombre_cluster = dict_clusters.get(cat, "-")
                nombre_cluster_2 = dict_clusters_2.get(cat, "-")
                
                ws.cell(row=row, column=col_detalle, value=cat).fill = color_fondo
                ws.cell(row=row, column=col_cluster, value=nombre_cluster).fill = color_fondo
                ws.cell(row=row, column=col_cluster2, value=nombre_cluster_2).fill = color_fondo
                ws.cell(row=row, column=col_score, value=score).fill = color_fondo
                ws.cell(row=row, column=col_debug, value=debug_str).fill = color_fondo

        # ==========================================
        # SOLUCIÓN AL ERROR DE REPARACIÓN (TABLAS XML)
        # ==========================================
        if hasattr(ws, 'tables'):
            for table_name in list(ws.tables.keys()):
                del ws.tables[table_name]

        # ==========================================
        # GUARDAR COMO COPIA
        # ==========================================
        directorio, nombre_archivo = os.path.split(ruta_archivo)
        nombre_base, ext = os.path.splitext(nombre_archivo)
        nueva_ruta = os.path.join(directorio, f"{nombre_base}_procesado{ext}")
        
        wb.save(nueva_ruta)

        # ==========================================
        # FASE 3: HOJAS ANALÍTICAS (PANDAS)
        # ==========================================
        try:
            print("Generando tablas dinámicas...")
            df = pd.read_excel(nueva_ruta, header=fila_cabecera-1)
            
            with pd.ExcelWriter(nueva_ruta, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                
                # Extraer Año de manera independiente para que ambas hojas puedan usarlo
                if 'FECHAREGISTRO' in df.columns:
                    df['Año'] = pd.to_datetime(df['FECHAREGISTRO'], errors='coerce').dt.year
                else:
                    df['Año'] = "Sin Año"

                # Hoja 1: Casos por Localidad y Año
                if 'SUCURSAL USUARIO' in df.columns:
                    df['Localidad_Limpia'] = df['SUCURSAL USUARIO'].apply(normalizar_localidad)
                    tabla_localidades = pd.crosstab(df['Localidad_Limpia'], df['Año'], margins=True, margins_name="Total")
                    
                    # ---------------- ORDENAR FILAS ----------------
                    orden_localidades = [
                        "01 - USAQUÉN", "02 - CHAPINERO", "03 - SANTA FE", "04 - SAN CRISTÓBAL", 
                        "05 - USME", "06 - TUNJUELITO", "07 - BOSA", "08 - KENNEDY", 
                        "09 - FONTIBÓN", "10 - ENGATIVÁ", "11 - SUBA", "12 - BARRIOS UNIDOS", 
                        "13 - TEUSAQUILLO", "14 - LOS MÁRTIRES", "15 - ANTONIO NARIÑO", 
                        "16 - PUENTE ARANDA", "17 - LA CANDELARIA", "18 - RAFAEL URIBE URIBE", 
                        "19 - CIUDAD BOLÍVAR", "20 - SUMAPAZ", "Total"
                    ]
                    # Filtramos el orden solo con las que existen para evitar NaNs
                    orden_presente = [loc for loc in orden_localidades if loc in tabla_localidades.index]
                    # Añadimos al final casos extraños como "SIN REGISTRO" o nombres raros
                    otras_loc = [loc for loc in tabla_localidades.index if loc not in orden_presente]
                    
                    tabla_localidades = tabla_localidades.reindex(orden_presente + otras_loc)
                    # -----------------------------------------------
                    
                    tabla_localidades.to_excel(writer, sheet_name='Casos por Localidad')
                else:
                    print("Aviso: No se generó Hoja 1 porque falta 'SUCURSAL USUARIO'.")
                
                # Hoja 2: Conteo de lista específica por año
                if 'Detalle Descripción' in df.columns and lista_conteo_personalizada:
                    def encontrar_mas_cercano(categoria):
                        if pd.isna(categoria) or categoria == "-": return "SIN CLASIFICAR"
                        matches = difflib.get_close_matches(str(categoria).upper(), lista_conteo_personalizada, n=1, cutoff=0.7)
                        return matches[0] if matches else "OTRO"
                        
                    df['Categoria_Lista'] = df['Detalle Descripción'].apply(encontrar_mas_cercano)
                    tabla_categorias = pd.crosstab(df['Categoria_Lista'], df['Año'], margins=True, margins_name="Total")
                    tabla_categorias.to_excel(writer, sheet_name='Conteo Categorías')
                else:
                    print("Aviso: No se generó Hoja 2 porque el TXT de lista estaba vacío o falta la columna descriptiva.")

                # Hoja 3: Conteo de Estados
                if 'ESTADO' in df.columns:
                    estados_validos = ['ANULADO', 'CERRADO', 'EN PROCESO', 'SOLUCIONADO', 'ASIGNADO']
                    df['Estado_Limpio'] = df['ESTADO'].astype(str).str.strip().str.upper()
                    df_estados = df[df['Estado_Limpio'].isin(estados_validos)]
                    tabla_estados = df_estados['Estado_Limpio'].value_counts().reset_index()
                    tabla_estados.columns = ['Estado', 'Cantidad']
                    tabla_estados.to_excel(writer, sheet_name='Conteo Estados', index=False)
                
                # Preparación de datos de localidad si existen
                if 'SUCURSAL USUARIO' in df.columns:
                    if 'Localidad_Limpia' not in df.columns:
                        df['Localidad_Limpia'] = df['SUCURSAL USUARIO'].apply(normalizar_localidad)
                    
                    # Hoja 4: Promedio de Días transcurridos
                    if 'Días transcurridos de solución' in df.columns:
                        df['Días_Num'] = df['Días transcurridos de solución'].astype(str).str.extract(r'(\d+)').astype(float)
                        tabla_promedio_dias = pd.pivot_table(
                            df, values='Días_Num', index='Localidad_Limpia', 
                            aggfunc='mean', margins=True, margins_name='PROMEDIO GENERAL'
                        )
                        tabla_promedio_dias.to_excel(writer, sheet_name='Promedio Días')
                        
                    # Hoja 5: Promedio de Tiempos totales
                    if 'Tiempos total de atención' in df.columns:
                        df['Tiempos_Num'] = df['Tiempos total de atención'].astype(str).str.extract(r'(\d+)').astype(float)
                        tabla_promedio_tiempos = pd.pivot_table(
                            df, values='Tiempos_Num', index='Localidad_Limpia', 
                            aggfunc='mean', margins=True, margins_name='PROMEDIO GENERAL'
                        )
                        tabla_promedio_tiempos.to_excel(writer, sheet_name='Promedio Tiempos')
                    
        except Exception as e:
            print(f"Error generando hojas analíticas: {e}")
        
        print(f"\n==========================================")
        if len(columnas_no_encontradas) == 0:
            print("¡Éxito Total! Todas las columnas y fórmulas se aplicaron correctamente.")
        else:
            print("Proceso a Medias. No se encontraron estas columnas:")
            for c in columnas_no_encontradas:
                print(f"- {c}")
        print(f"Ruta: {nueva_ruta}")
        print(f"==========================================\n")

    except Exception as e:
        print(f"Ocurrió un error inesperado:\n{str(e)}")

if __name__ == "__main__":
    procesar_excel()
    

Paso 1: Seleccionando archivo TXT de categorías...
Paso 1.1: Seleccionando archivo TXT de Clústeres...
Paso 1.2: Seleccionando Excel de mapeo de Clústeres 2...
Paso 1.3: Seleccionando archivo TXT con lista de conteo (Tercera Hoja)...
Paso 2: Seleccionando archivo Excel a procesar...
Procesando Excel...
Generando tablas dinámicas...

¡Éxito Total! Todas las columnas y fórmulas se aplicaron correctamente.
Ruta: C:/Users/Pc/OneDrive/Trabajo/2026/Secretaria de Gobierno/05 Mayo/Reporte de analisis Casos HOLA\Reporte General de Casos Sipse Localidades 1 Ene al 29 Sep del 2026_procesado.xlsx

